# Demonstration: Creating a Small Transformer using PyTorch

## Scenario:
- RetailVerse, a growing e-commerce platform, receives thousands of customer reviews each month across its diverse product catalog. The company's support and product teams aim to quickly assess customer sentiment to prioritize feature improvements, resolve complaints efficiently, and highlight positive trends. However, the existing process of manually labeling sentiment in reviews is time-consuming, inconsistent, and not scalable with the company's growth.

- To address this challenge, RetailVerse's machine learning team has been assigned the task of developing a lightweight, transparent, and self-contained sentiment analysis model that does not depend on large-scale external APIs. As an initial step, the team chooses to implement a mini transformer model using PyTorch, trained on internally curated customer review data.

- The objective of this initiative is to classify each review into one of three sentiment categories—positive, neutral, or negative, with reliable accuracy, using a compact transformer architecture built from scratch and trained on a controlled dataset.

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import pandas as pd
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Upload and Load the Dataset

Download the **sentiment_dataset.csv** from the LMS

In [ ]:
uploaded = files.upload()
df = pd.read_csv("sentiment_dataset_200.csv")
print("Sample data:")
print(df.head())

Saving sentiment_dataset.csv to sentiment_dataset (2).csv
Sample data:
                   review sentiment
0      Regret buying this  negative
1    Very poor experience  negative
2      Regret buying this  negative
3     Mediocre experience   neutral
4  Broke down in two days  negative


## Tokenization and Vocabulary

In [ ]:
def tokenize(text): return text.lower().split()

vocab = sorted(set(word for sentence in df["review"] for word in tokenize(sentence)))
word2idx = {word: i+4 for i, word in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
word2idx["<CLS>"] = 2
word2idx["<EOS>"] = 3
idx2word = {v: k for k, v in word2idx.items()}
vocab_size = len(word2idx)

def encode(sentence, max_len=15):
    tokens = [word2idx.get(w, 1) for w in tokenize(sentence)]
    tokens = [2] + tokens[:max_len-2] + [3]
    while len(tokens) < max_len:
        tokens.append(0)
    return tokens

X = torch.tensor([encode(s, 15) for s in df["review"]]).to(device)
label2idx = {"negative": 0, "neutral": 1, "positive": 2}
classes = ["negative", "neutral", "positive"]
y = torch.tensor([label2idx[s] for s in df["sentiment"]]).to(device)

## Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=15):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (2*i/d_model)))
                if i + 1 < d_model:
                    pe[pos, i+1] = math.cos(pos / (10000 ** (2*i/d_model)))
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

##Mini Transformer Encoder

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.heads = heads
        self.d_k = d_model // heads
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x).view(B, T, 3, self.heads, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out(context)

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attn = SelfAttention(d_model, heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_model*4), nn.ReLU(), nn.Linear(d_model*4, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))
        return self.norm2(x + self.ff(x))

class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, heads=4, max_len=15, num_classes=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.encoder = TransformerEncoder(d_model, heads)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos(x)
        x = self.encoder(x)
        cls_token = x[:, 0]  # [CLS]
        return self.classifier(cls_token)

## Train the Model

In [ ]:
model = SentimentClassifier(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(300):
    model.train()
    logits = model(X)
    loss = criterion(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 30 == 0:
        pred = torch.argmax(logits, dim=1)
        acc = (pred == y).float().mean()
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Accuracy={acc:.2f}")

Epoch 30: Loss=0.0242, Accuracy=1.00
Epoch 60: Loss=0.0018, Accuracy=1.00
Epoch 90: Loss=0.0011, Accuracy=1.00
Epoch 120: Loss=0.0009, Accuracy=1.00
Epoch 150: Loss=0.0008, Accuracy=1.00
Epoch 180: Loss=0.0007, Accuracy=1.00
Epoch 210: Loss=0.0006, Accuracy=1.00
Epoch 240: Loss=0.0005, Accuracy=1.00
Epoch 270: Loss=0.0005, Accuracy=1.00
Epoch 300: Loss=0.0004, Accuracy=1.00


##Sentiment Prediction

In [ ]:
def predict_sentiment(sentence):
    model.eval()
    with torch.no_grad():
        encoded = torch.tensor([encode(sentence)]).to(device)
        logits = model(encoded)
        pred = torch.argmax(logits, dim=1).item()
        return classes[pred]

##Inference Function

In [ ]:
while True:
    review = input("\nEnter a review (or 'exit'): ")
    if review.lower() == 'exit':
        break
    print("Predicted Sentiment:", predict_sentiment(review))


Enter a review (or 'exit'): Terrible Product
Predicted Sentiment: negative

Enter a review (or 'exit'): Great value for money
Predicted Sentiment: positive

Enter a review (or 'exit'): Fantastic
Predicted Sentiment: positive

Enter a review (or 'exit'): Satisfcatory use
Predicted Sentiment: neutral

Enter a review (or 'exit'): exit


# Sample Inputs
1. Loved the product, Predicted Sentiment: positive

2. Not good, Predicted Sentiment: negative

3. Great, Predicted Sentiment: positive

4. Great value for the price, Predicted Sentiment: positive

5. Just a basic item, Predicted Sentiment: neutral

6. So-so, Predicted Sentiment: neutral

7. Average performance, Predicted Sentiment: neutral

8. Terrible product, Predicted Sentiment: negative

9. Regret buying this, Predicted Sentiment: negative


**Note:** This model may not always predict sentiment accurately due to limited training data, lack of pretrained language understanding, and difficulty in handling very short, ambiguous, or sarcastic inputs.
